# NTU RGB+D 60 — Data Exploration & Preprocessing
**Dataset:** ntu60_hrnet.pkl (PYSKL format, HRNet 2D keypoints)  
**Split:** Cross-Subject (xsub)

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'ntu60_hrnet.pkl/ntu60_hrnet.pkl'

## 1. Veriyi Yükle

In [ ]:
with open(DATA_PATH, 'rb') as f:
    raw = pickle.load(f)

annotations = raw['annotations']
split       = raw['split']

print(f"Toplam sample sayısı : {len(annotations):,}")
print(f"Split anahtarları    : {list(split.keys())}")
for k, v in split.items():
    print(f"  {k:20s}: {len(v):,} sample")

In [ ]:
# İlk annotation'a bak
ann = annotations[0]
print("frame_dir     :", ann['frame_dir'])
print("label         :", ann['label'])
print("total_frames  :", ann['total_frames'])
print("keypoint shape:", ann['keypoint'].shape,  "  → (kişi, frame, joint, xy)")
print("score shape   :", ann['keypoint_score'].shape)

## 2. Temel İstatistikler

In [ ]:
frame_lengths = np.array([a['total_frames'] for a in annotations])
labels_all    = np.array([a['label']        for a in annotations])
person_counts = np.array([a['keypoint'].shape[0] for a in annotations])

print("=== Frame Uzunlukları ===")
print(f"  min  : {frame_lengths.min()}")
print(f"  max  : {frame_lengths.max()}")
print(f"  mean : {frame_lengths.mean():.1f}")
print(f"  %50  : {np.percentile(frame_lengths, 50):.0f}")
print(f"  %95  : {np.percentile(frame_lengths, 95):.0f}")

print("\n=== Kişi Sayısı ===")
for k, v in sorted(Counter(person_counts).items()):
    print(f"  {k} kişi: {v:,} sample ({100*v/len(annotations):.1f}%)")

print("\n=== Label ===")
print(f"  unique class sayısı : {len(set(labels_all))}")
label_counts = Counter(labels_all)
print(f"  min sample/class    : {min(label_counts.values())}")
print(f"  max sample/class    : {max(label_counts.values())}")
print(f"  ortalama/class      : {np.mean(list(label_counts.values())):.0f}")

## 3. Görselleştirmeler

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Frame uzunluğu dağılımı
axes[0].hist(frame_lengths, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(frame_lengths.mean(), color='red', linestyle='--', label=f'Ortalama={frame_lengths.mean():.0f}')
axes[0].set_title('Frame Uzunluğu Dağılımı')
axes[0].set_xlabel('Frame sayısı')
axes[0].set_ylabel('Frekans')
axes[0].legend()

# Label dağılımı
counts = [label_counts[i] for i in range(60)]
axes[1].bar(range(60), counts, color='steelblue', width=0.8)
axes[1].set_title('Class Başına Sample Sayısı (60 class)')
axes[1].set_xlabel('Action Class ID')
axes[1].set_ylabel('Sample sayısı')

# Kişi sayısı dağılımı
pc_counts = Counter(person_counts)
axes[2].bar(pc_counts.keys(), pc_counts.values(), color='steelblue')
axes[2].set_title('Sample Başına Kişi Sayısı')
axes[2].set_xlabel('Kişi sayısı')
axes[2].set_ylabel('Sample sayısı')

plt.tight_layout()
plt.savefig('stats.png', dpi=120)
plt.show()

## 4. Skeleton Görselleştirme
COCO 17-joint skeleton: burun, gözler, kulaklar, omuzlar, dirsekler, bilekler, kalçalar, dizler, ayak bilekleri

In [ ]:
# COCO 17-joint bağlantıları
COCO_EDGES = [
    (0, 1), (0, 2),           # burun → gözler
    (1, 3), (2, 4),           # gözler → kulaklar
    (5, 6),                   # omuz-omuz
    (5, 7), (7, 9),           # sol kol
    (6, 8), (8, 10),          # sağ kol
    (5, 11), (6, 12),         # gövde
    (11, 12),                 # kalça-kalça
    (11, 13), (13, 15),       # sol bacak
    (12, 14), (14, 16),       # sağ bacak
]

JOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

# NTU60 action isimleri (0-59)
NTU60_CLASSES = [
    'drink water', 'eat meal/snack', 'brushing teeth', 'brushing hair', 'drop',
    'pickup', 'throw', 'sitting down', 'standing up', 'clapping',
    'reading', 'writing', 'tear up paper', 'wear jacket', 'take off jacket',
    'wear a shoe', 'take off a shoe', 'wear on glasses', 'take off glasses', 'put on a hat/cap',
    'take off a hat/cap', 'cheer up', 'hand waving', 'kicking something', 'reach into pocket',
    'hopping (one foot jumping)', 'jump up', 'make a phone call/answer phone', 'playing with phone/tablet', 'typing on a keyboard',
    'pointing to something with finger', 'taking a selfie', 'check time (from watch)', 'rub two hands together', 'nod head/bow',
    'shake head', 'wipe face', 'salute', 'put the palms together', 'cross hands in front',
    'sneeze/cough', 'staggering', 'falling', 'touch head (headache)', 'touch chest (stomachache/heart pain)',
    'touch back (backache)', 'touch neck (neckache)', 'nausea or vomiting condition', 'use a fan (with hand or paper)/feeling warm', 'punching/slapping other person',
    'kicking other person', 'pushing other person', 'pat on back of other person', 'point finger at the other person', 'hugging other person',
    'giving something to other person', 'touch other persons pocket', 'handshaking', 'walking towards each other', 'walking apart from each other'
]

def draw_skeleton(ax, keypoints, scores=None, threshold=0.3, title=''):
    """keypoints: (17, 2) array — x, y koordinatları"""
    for (j1, j2) in COCO_EDGES:
        if scores is not None and (scores[j1] < threshold or scores[j2] < threshold):
            continue
        x = [keypoints[j1, 0], keypoints[j2, 0]]
        y = [keypoints[j1, 1], keypoints[j2, 1]]
        ax.plot(x, y, 'b-', linewidth=2, alpha=0.7)
    ax.scatter(keypoints[:, 0], keypoints[:, 1], c='red', s=20, zorder=5)
    ax.set_title(title, fontsize=9)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.axis('off')

In [ ]:
# 12 farklı action'dan örnek skeleton çiz (orta frame)
sample_classes = list(range(0, 60, 5))  # 0, 5, 10, ..., 55

# Her class'tan bir sample bul
class_to_sample = {}
for ann in annotations:
    lbl = ann['label']
    if lbl in sample_classes and lbl not in class_to_sample:
        class_to_sample[lbl] = ann
    if len(class_to_sample) == len(sample_classes):
        break

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, cls_id in enumerate(sorted(class_to_sample.keys())):
    ann = class_to_sample[cls_id]
    mid_frame = ann['total_frames'] // 2
    kp     = ann['keypoint'][0, mid_frame]       # (17, 2)
    scores = ann['keypoint_score'][0, mid_frame]  # (17,)
    title  = f"[{cls_id}] {NTU60_CLASSES[cls_id]}"
    draw_skeleton(axes[idx], kp, scores, title=title)

plt.suptitle('Örnek Skeleton Pozu — Her Action Class\'tan Bir Orta Frame', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('sample_skeletons.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Keypoint Kalite Analizi

In [ ]:
# Tüm score'ları örnekle (her 10 sample'da bir)
sampled_scores = []
for ann in annotations[::10]:
    sampled_scores.append(ann['keypoint_score'].flatten())
all_scores = np.concatenate(sampled_scores)

print("=== Keypoint Score İstatistikleri ===")
for pct in [1, 5, 10, 25, 50]:
    print(f"  %{pct:2d} percentile : {np.percentile(all_scores, pct):.4f}")
print(f"  mean           : {all_scores.mean():.4f}")
print(f"  <0.3 oranı     : {(all_scores < 0.3).mean()*100:.2f}%  (düşük güven)")

plt.figure(figsize=(8, 4))
plt.hist(all_scores, bins=50, color='steelblue', edgecolor='white')
plt.axvline(0.3, color='red', linestyle='--', label='Eşik=0.3')
plt.title('Keypoint Score Dağılımı (sampled)')
plt.xlabel('Confidence score')
plt.ylabel('Frekans')
plt.legend()
plt.tight_layout()
plt.savefig('keypoint_scores.png', dpi=120)
plt.show()

## 6. Normalizasyon
Tüm skeleton'ları **kalça merkezine** göre ortala, **torso çapına** göre ölçeklendir.

In [ ]:
# COCO joint indeksleri
LEFT_HIP  = 11
RIGHT_HIP = 12
LEFT_SHOULDER  = 5
RIGHT_SHOULDER = 6

def normalize_skeleton(keypoint):
    """
    keypoint: (M, T, V, C) — M:kişi, T:frame, V:17 joint, C:2 (xy)
    Dönüş: normalize edilmiş (M, T, V, C)
    """
    kp = keypoint.astype(np.float32)
    # Kalça merkezi (her frame için)
    hip_center = (kp[:, :, LEFT_HIP, :] + kp[:, :, RIGHT_HIP, :]) / 2.0  # (M, T, C)
    kp -= hip_center[:, :, np.newaxis, :]  # merkeze al

    # Torso çapı: omuz orta → kalça ortası mesafesi
    shoulder_center = (kp[:, :, LEFT_SHOULDER, :] + kp[:, :, RIGHT_SHOULDER, :]) / 2.0
    torso_size = np.linalg.norm(shoulder_center, axis=-1, keepdims=True)  # (M, T, 1)
    torso_size = np.maximum(torso_size, 1e-6)  # sıfıra bölünmeyi önle
    scale = torso_size[:, :, :, np.newaxis]  # (M, T, 1, 1)
    kp /= scale
    return kp

# Test
test_ann = annotations[100]
kp_raw  = test_ann['keypoint'].astype(np.float32)  # (1, T, 17, 2)
kp_norm = normalize_skeleton(kp_raw)

print("Ham    — kalça merkezi (orta frame):", kp_raw[0, kp_raw.shape[1]//2, LEFT_HIP, :].round(1))
print("Norm   — kalça merkezi (orta frame):", kp_norm[0, kp_norm.shape[1]//2, LEFT_HIP, :].round(3))
print("Norm range: [{:.3f}, {:.3f}]".format(kp_norm.min(), kp_norm.max()))

In [ ]:
# Ham vs Normalize karşılaştırması
mid = kp_raw.shape[1] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
draw_skeleton(axes[0], kp_raw[0, mid], title='Ham Koordinatlar')
draw_skeleton(axes[1], kp_norm[0, mid], title='Normalize Edilmiş')
fig.suptitle(f"Action: {NTU60_CLASSES[test_ann['label']]}")
plt.tight_layout()
plt.savefig('normalization_compare.png', dpi=120)
plt.show()

## 7. Padding / Sampling — Sabit Frame Uzunluğu
ST-GCN sabit T uzunluğu bekler. Frame sayısı T'den kısaysa sıfır pad, uzunsa uniform sample.

In [ ]:
TARGET_FRAMES = 100  # %95 percentile ~100 frame altında

def pad_or_sample(keypoint, target_t=TARGET_FRAMES):
    """
    keypoint: (M, T, V, C)
    Dönüş  : (M, target_t, V, C)
    """
    M, T, V, C = keypoint.shape
    if T == target_t:
        return keypoint
    elif T < target_t:
        pad = np.zeros((M, target_t - T, V, C), dtype=keypoint.dtype)
        return np.concatenate([keypoint, pad], axis=1)
    else:
        indices = np.linspace(0, T - 1, target_t, dtype=int)
        return keypoint[:, indices, :, :]

# Test
for lbl, frames_needed in [('kısa (32f)', 32), ('orta (84f)', 84), ('uzun (300f)', 300)]:
    dummy = np.random.randn(1, frames_needed, 17, 2).astype(np.float32)
    out = pad_or_sample(dummy)
    print(f"  {lbl:15s} → {dummy.shape} → {out.shape}")

## 8. Preprocessing Pipeline — Tam Veriyi İşle
Bu adım biraz zaman alabilir (~5 dk). Sonucu `processed_xsub.npz` olarak kaydeder.

In [ ]:
from tqdm.auto import tqdm

TARGET_FRAMES = 100
MAX_PERSONS   = 2   # NTU'da max 2 kişi

def process_split(ann_list, split_name):
    """
    ann_list : annotation dict listesi
    Dönüş    : X=(N, MAX_PERSONS, TARGET_FRAMES, 17, 2), y=(N,)
    """
    # frame_dir → annotation index map
    fd_to_ann = {a['frame_dir']: a for a in annotations}

    X_list, y_list = [], []
    missing = 0
    for fd in tqdm(ann_list, desc=split_name):
        if fd not in fd_to_ann:
            missing += 1
            continue
        ann = fd_to_ann[fd]
        kp  = ann['keypoint'].astype(np.float32)  # (M, T, 17, 2)
        M, T, V, C = kp.shape

        # Normalize et
        kp = normalize_skeleton(kp)

        # Sabit frame uzunluğuna getir
        kp = pad_or_sample(kp, TARGET_FRAMES)     # (M, 100, 17, 2)

        # MAX_PERSONS'a pad/truncate
        if M < MAX_PERSONS:
            pad_p = np.zeros((MAX_PERSONS - M, TARGET_FRAMES, V, C), dtype=np.float32)
            kp = np.concatenate([kp, pad_p], axis=0)
        else:
            kp = kp[:MAX_PERSONS]

        X_list.append(kp)           # (2, 100, 17, 2)
        y_list.append(ann['label'])

    if missing:
        print(f"  Uyarı: {missing} sample bulunamadı.")

    X = np.stack(X_list)            # (N, 2, 100, 17, 2)
    y = np.array(y_list)
    return X, y

X_train, y_train = process_split(split['xsub_train'], 'xsub_train')
X_val,   y_val   = process_split(split['xsub_val'],   'xsub_val')

print(f"\nX_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}    y_val  : {y_val.shape}")

In [ ]:
np.savez_compressed(
    'processed_xsub.npz',
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val
)
print("processed_xsub.npz kaydedildi.")
import os
size_mb = os.path.getsize('processed_xsub.npz') / 1e6
print(f"Dosya boyutu: {size_mb:.1f} MB")

## 9. Özet

| | Train | Val |
|---|---|---|
| Sample sayısı | `y_train.shape[0]` | `y_val.shape[0]` |
| Tensor shape | `(N, 2, 100, 17, 2)` | `(N, 2, 100, 17, 2)` |
| Normalizasyon | Kalça merkezi + torso scale | |
| Frame | Pad/sample → 100 | |

**Sonraki adım:** `02_model_training.ipynb` — ST-GCN modelini tanımla ve eğit.

In [ ]:
# Hızlı doğrulama
d = np.load('processed_xsub.npz')
print("Yüklenen shape'ler:")
for k in d.files:
    print(f"  {k}: {d[k].shape}")

print("\nClass dağılımı (train, ilk 10):")
train_counts = Counter(d['y_train'])
for cls_id in range(10):
    print(f"  [{cls_id:2d}] {NTU60_CLASSES[cls_id]:35s}: {train_counts[cls_id]}")